# LLM Server Smoke Test

Tests the `/v1/process` endpoint against the updated `BrainInstruction` schema.

Command types covered:
- `vel` — velocity control (x / y / yaw / h)
- `brake` — smooth stop
- `estop` — emergency stop
- `cmd` — named brainstem commands (`StandSitToggle`, `Hello`, `Twist`, `Moonwalk`, `TwistJump`, `ResetToZero`, gaits, modes)

## 1) Configure server URL

In [6]:
SERVER_URL    = "http://localhost:8000"
PROCESS_URL   = f"{SERVER_URL}/v1/process"
HEALTH_URL    = f"{SERVER_URL}/health"
SERVER_API_KEY = "@RAI026LLM"

SERVER_URL

'http://localhost:8000'

## 2) Imports

In [7]:
import json
import urllib.request
import urllib.error

## 3) Health check

In [8]:
def get_health(url: str) -> dict:
    req = urllib.request.Request(url, method="GET")
    with urllib.request.urlopen(req, timeout=10) as resp:
        return json.loads(resp.read().decode("utf-8"))

get_health(HEALTH_URL)

{'status': 'ok', 'provider': 'ollama', 'model': 'phi4:14b'}

## 4) Helper functions

In [9]:
def process_transcript(
    transcript: str,
    context: dict | None = None,
    history: list | None = None,
    provider: str | None = None,
    model: str | None = None,
) -> dict:
    payload = {
        "transcript": transcript,
        "context": context or {},
        "history": history or [],
    }
    if provider:
        payload["provider"] = provider
    if model:
        payload["model"] = model

    data = json.dumps(payload).encode("utf-8")
    headers = {"Content-Type": "application/json"}
    if SERVER_API_KEY:
        headers["X-API-Key"] = SERVER_API_KEY

    req = urllib.request.Request(PROCESS_URL, data=data, headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=150) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        return {"error": f"HTTP {e.code}", "body": body}


def fmt_cmd(instruction: dict) -> str:
    cmd = instruction.get("command", {})
    t = cmd.get("type", "?")
    if t == "vel":
        return f"vel(x={cmd.get('x')}, y={cmd.get('y')}, yaw={cmd.get('yaw')}, h={cmd.get('h', 0)})"
    if t == "cmd":
        return f"cmd:{cmd.get('code')}"
    return t  # brake / estop


def show(label: str, resp: dict) -> None:
    instr = resp.get("instruction", {})
    err   = resp.get("error")
    if err:
        print(f"  ERROR: {err} — {resp.get('body', '')[:200]}")
        return
    print(f"  transcript : {label}")
    print(f"  command    : {fmt_cmd(instr)}")
    print(f"  reasoning  : {instr.get('reasoning')}")
    print(f"  confidence : {instr.get('confidence')}")

## 4b) Debug: single call — inspect raw model output

Run this cell alone first. It sends "hi, how are you?" and prints the full server response (or the full error body if it fails).

In [10]:
_payload = json.dumps({
    "transcript": "hi, how are you?",
    "context": {},
    "history": [],
}).encode("utf-8")

_req = urllib.request.Request(
    PROCESS_URL,
    data=_payload,
    headers={"Content-Type": "application/json", "X-API-Key": SERVER_API_KEY},
    method="POST",
)

try:
    with urllib.request.urlopen(_req, timeout=150) as _resp:
        _body = _resp.read().decode("utf-8")
    print("✓ SUCCESS — full response:")
    print(json.dumps(json.loads(_body), indent=2))
except urllib.error.HTTPError as _e:
    _body = _e.read().decode("utf-8", errors="replace")
    print(f"✗ HTTP {_e.code} — full error body:")
    print(_body)

✓ SUCCESS — full response:
{
  "transcript": "hi, how are you?",
  "provider": "ollama",
  "model": "phi4:14b",
  "instruction": {
    "command": {
      "type": "brake"
    },
    "reasoning": "Greeting detected; no action needed.",
    "confidence": 0.9
  },
  "raw_model_output": "{\n  \"command\": {\"type\":\"brake\"},\n  \"reasoning\": \"Greeting detected; no action needed.\",\n  \"confidence\": 0.9\n}"
}


## 5) Velocity commands (`vel`)

Expected: `command.type == "vel"` with reasonable x / y / yaw values.

In [11]:
vel_cases = [
    ("move forward",          {"robot_state": "standing"}),
    ("go backward slowly",    {"robot_state": "standing"}),
    ("strafe left",           {"robot_state": "standing"}),
    ("turn right",            {"robot_state": "standing"}),
    ("move forward and turn left slightly", {"robot_state": "standing"}),
]

print("=== VEL COMMANDS ===\n")
for transcript, ctx in vel_cases:
    resp = process_transcript(transcript, context=ctx)
    show(transcript, resp)
    print()

=== VEL COMMANDS ===

  transcript : move forward
  command    : vel(x=1.0, y=0.0, yaw=0.0, h=0.0)
  reasoning  : User wants to move forward; robot is standing.
  confidence : 1.0

  transcript : go backward slowly
  command    : vel(x=-0.2, y=0.0, yaw=0.0, h=0.0)
  reasoning  : User wants to move backward slowly; set x velocity to -0.2.
  confidence : 1.0

  transcript : strafe left
  command    : vel(x=-0.5, y=0.0, yaw=0.0, h=0.0)
  reasoning  : Strafe left corresponds to negative x velocity.
  confidence : 1.0

  transcript : turn right
  command    : vel(x=0.0, y=0.0, yaw=0.5, h=0.0)
  reasoning  : User wants to turn right; yaw set positive for clockwise rotation.
  confidence : 1.0

  transcript : move forward and turn left slightly
  command    : vel(x=0.7, y=-0.3, yaw=-0.3, h=0.0)
  reasoning  : Move forward and turn left slightly in standing state.
  confidence : 1.0



## 6) Brake & E-Stop

Expected: `brake` for gentle stops, `estop` for danger/emergency triggers.

In [12]:
stop_cases = [
    ("stop gently",              {}),
    ("slow down and wait",       {}),
    ("emergency stop now",       {}),
    ("danger, stop immediately", {}),
    ("halt, someone is in the way", {}),
]

print("=== BRAKE / ESTOP ===\n")
for transcript, ctx in stop_cases:
    resp = process_transcript(transcript, context=ctx)
    show(transcript, resp)
    print()

=== BRAKE / ESTOP ===

  transcript : stop gently
  command    : brake
  reasoning  : User requested a gentle stop.
  confidence : 0.9

  transcript : slow down and wait
  command    : brake
  reasoning  : User requested to slow down and wait.
  confidence : 0.9

  transcript : emergency stop now
  command    : estop
  reasoning  : User requested an emergency stop.
  confidence : 1.0

  transcript : danger, stop immediately
  command    : estop
  reasoning  : User indicated danger; immediate halt required.
  confidence : 1.0

  transcript : halt, someone is in the way
  command    : estop
  reasoning  : User indicated 'halt' and mentioned danger.
  confidence : 1.0



## 7) Named commands (`cmd`)

Expected: `command.type == "cmd"` and `command.code` matching the intended `CmdCode`.

In [13]:
# (transcript, context, expected_code)
cmd_cases = [
    ("sit down",                   {"robot_state": "standing"}, "StandSitToggle"),
    ("stand up",                   {"robot_state": "sitting"},  "StandSitToggle"),
    ("switch to move mode",        {"robot_state": "standing"}, "MoveMode"),
    ("switch to pose mode",        {"robot_state": "standing"}, "PoseMode"),
    ("reset all joints to zero",   {"robot_state": "standing"}, "ResetToZero"),
    ("use slow gait",              {"robot_state": "standing"}, "GaitFlatSlow"),
    ("use medium speed gait",      {"robot_state": "standing"}, "GaitFlatMedium"),
    ("use fast gait",              {"robot_state": "standing"}, "GaitFlatFast"),
]

print("=== NAMED COMMANDS ===\n")
for transcript, ctx, expected in cmd_cases:
    resp = process_transcript(transcript, context=ctx)
    instr = resp.get("instruction", {})
    cmd   = instr.get("command", {})
    got   = cmd.get("code", cmd.get("type", "?"))
    mark  = "✓" if got == expected else f"✗ (expected {expected})"
    print(f"  [{mark}] {transcript!r}")
    print(f"         → {fmt_cmd(instr)}  | conf={instr.get('confidence')}")
    print()

=== NAMED COMMANDS ===

  [✓] 'sit down'
         → cmd:StandSitToggle  | conf=1.0

  [✓] 'stand up'
         → cmd:StandSitToggle  | conf=1.0

  [✓] 'switch to move mode'
         → cmd:MoveMode  | conf=1.0

  [✓] 'switch to pose mode'
         → cmd:PoseMode  | conf=1.0

  [✓] 'reset all joints to zero'
         → cmd:ResetToZero  | conf=1.0

  [✓] 'use slow gait'
         → cmd:GaitFlatSlow  | conf=1.0

  [✓] 'use medium speed gait'
         → cmd:GaitFlatMedium  | conf=1.0

  [✓] 'use fast gait'
         → cmd:GaitFlatFast  | conf=1.0



## 8) Precondition tests

The LLM should respect `robot_state` from context:
- `Hello` only when **sitting**
- `Twist`, `Moonwalk`, `TwistJump` only when **standing**

In [16]:
# (transcript, context, expected_code, note)
precondition_cases = [
    # Hello requires sitting
    ("say hello to everyone",  {"robot_state": "sitting"},  "Hello",          "sitting ✓ — should output Hello"),
    ("say hello to everyone",  {"robot_state": "standing"}, "StandSitToggle", "standing ✗ — should sit first, not Hello"),
    # Twist/Moonwalk/TwistJump require standing
    ("do a twist",             {"robot_state": "standing"}, "Twist",          "standing ✓ — should output Twist"),
    ("do a twist",             {"robot_state": "sitting"},  "StandSitToggle", "sitting ✗ — should stand first, not Twist"),
    ("moonwalk now",           {"robot_state": "standing"}, "Moonwalk",       "standing ✓ — should output Moonwalk"),
    ("do a twist jump",        {"robot_state": "standing"}, "TwistJump",      "standing ✓ — should output TwistJump"),
]

print("=== PRECONDITION TESTS ===\n")
for transcript, ctx, expected, note in precondition_cases:
    resp  = process_transcript(transcript, context=ctx)
    instr = resp.get("instruction", {})
    cmd   = instr.get("command", {})
    got   = cmd.get("code", cmd.get("type", "?"))
    mark  = "✓" if got == expected else f"✗ (got {got!r}, expected {expected!r})"
    print(f"  [{mark}]  {note}")
    print(f"           → {fmt_cmd(instr)}  | conf={instr.get('confidence')}")
    print(f"           reasoning: {instr.get('reasoning')}")
    print()

=== PRECONDITION TESTS ===

  [✓]  sitting ✓ — should output Hello
           → cmd:Hello  | conf=1.0
           reasoning: Robot is sitting, precondition met for Hello command.

  [✓]  standing ✗ — should sit first, not Hello
           → cmd:StandSitToggle  | conf=0.6
           reasoning: Hello requires sitting; robot is standing.

  [✓]  standing ✓ — should output Twist
           → cmd:Twist  | conf=1.0
           reasoning: Robot is standing, Twist command can be executed.

  [✓]  sitting ✗ — should stand first, not Twist
           → cmd:StandSitToggle  | conf=0.6
           reasoning: Twist requires standing; robot is sitting.

  [✓]  standing ✓ — should output Moonwalk
           → cmd:Moonwalk  | conf=1.0
           reasoning: Robot is standing, precondition met for Moonwalk.

  [✓]  standing ✓ — should output TwistJump
           → cmd:TwistJump  | conf=1.0
           reasoning: Robot is standing, TwistJump precondition met.



## 9) Full batch smoke test

Runs every supported command type in one pass and prints a pass/fail summary.
A test **passes** if the response parses correctly and `command.type` is present.

In [17]:
VALID_TYPES = {"vel", "brake", "estop", "cmd"}

all_cases = [
    # vel
    ("move forward at medium speed",      {"robot_state": "standing"}),
    ("go left",                           {"robot_state": "standing"}),
    ("spin clockwise",                    {"robot_state": "standing"}),
    # brake / estop
    ("stop gently",                       {}),
    ("emergency stop",                    {}),
    # cmd — stand/sit
    ("stand up",                          {"robot_state": "sitting"}),
    ("sit down",                          {"robot_state": "standing"}),
    # cmd — modes
    ("enter move mode",                   {"robot_state": "standing"}),
    ("enter pose mode",                   {"robot_state": "standing"}),
    # cmd — gestures / tricks
    ("say hello",                         {"robot_state": "sitting"}),
    ("do a twist",                        {"robot_state": "standing"}),
    ("moonwalk",                          {"robot_state": "standing"}),
    ("twist jump",                        {"robot_state": "standing"}),
    # cmd — gaits / reset
    ("use slow gait",                     {"robot_state": "standing"}),
    ("use medium gait",                   {"robot_state": "standing"}),
    ("use fast gait",                     {"robot_state": "standing"}),
    ("reset to zero",                     {"robot_state": "standing"}),
]

passed, failed = 0, 0
print(f"{'#':<3} {'RESULT':<6} {'TYPE':<8} {'CODE/DETAIL':<20} TRANSCRIPT")
print("-" * 75)

for i, (transcript, ctx) in enumerate(all_cases, 1):
    resp  = process_transcript(transcript, context=ctx)
    instr = resp.get("instruction", {})
    cmd   = instr.get("command", {})
    t     = cmd.get("type", "")
    code  = cmd.get("code", "")
    ok    = t in VALID_TYPES and not resp.get("error")

    if ok:
        passed += 1
        mark = "PASS"
    else:
        failed += 1
        mark = "FAIL"

    detail = code if code else t
    print(f"{i:<3} {mark:<6} {t:<8} {detail:<20} {transcript!r}")

print("-" * 75)
print(f"\nResults: {passed} passed, {failed} failed out of {len(all_cases)} tests.")

#   RESULT TYPE     CODE/DETAIL          TRANSCRIPT
---------------------------------------------------------------------------
1   PASS   cmd      MoveMode             'move forward at medium speed'
2   PASS   vel      vel                  'go left'
3   PASS   cmd      Twist                'spin clockwise'
4   PASS   brake    brake                'stop gently'
5   PASS   estop    estop                'emergency stop'
6   PASS   cmd      StandSitToggle       'stand up'
7   PASS   cmd      StandSitToggle       'sit down'
8   PASS   cmd      MoveMode             'enter move mode'
9   PASS   cmd      PoseMode             'enter pose mode'
10  PASS   cmd      Hello                'say hello'
11  PASS   cmd      Twist                'do a twist'
12  PASS   cmd      Moonwalk             'moonwalk'
13  PASS   cmd      TwistJump            'twist jump'
14  PASS   cmd      GaitFlatSlow         'use slow gait'
15  PASS   cmd      GaitFlatMedium       'use medium gait'
16  PASS   cmd      GaitFla